In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip"] + list(args))

# ── P100-compatible PyTorch (sm_60 kernels) ───────────────────────────────────
# cu117 wheel index has been retired; cu118 wheels still include sm_60 kernels.
print("Installing P100-compatible PyTorch for sm_60 (cu118 wheel)...")
try:
    pip("install", "--quiet", "--upgrade",
        "torch==2.0.1+cu118", "torchvision==0.15.2+cu118",
        "--extra-index-url", "https://download.pytorch.org/whl/cu118")
    print("Installed torch 2.0.1+cu118 (sm_60 compatible).")
except subprocess.CalledProcessError:
    print("cu118 index failed — falling back to latest stable torch...")
    pip("install", "--quiet", "--upgrade", "torch", "torchvision",
        "--extra-index-url", "https://download.pytorch.org/whl/cu118")
    print("Installed latest torch via cu118 index.")

pip("install", "--quiet",
    "transformers==4.40.2", "timm==0.9.16",
    "scikit-learn", "pandas", "numpy", "matplotlib", "imbalanced-learn", "Pillow")
print("All deps installed.")

# ── Imports ───────────────────────────────────────────────────────────────────
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, pandas as pd, os, json, warnings, copy
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE as _SMOTE
import torchvision.models as tvm
import torchvision.transforms as T
from transformers import (CLIPModel, CLIPProcessor,
                          ViTModel, ViTImageProcessor)
warnings.filterwarnings('ignore')

print(f"torch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Smoke-test sm_60 kernels actually work
if torch.cuda.is_available():
    try:
        x = torch.randn(4, 4, device='cuda')
        _ = (x @ x).sum()
        print("sm_60 kernel smoke-test: OK")
    except Exception as e:
        print(f"WARNING: CUDA kernel error: {e}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Constants ─────────────────────────────────────────────────────────────────
BASE_SEED    = 42
SEEDS        = [42, 0, 1, 7, 99]
N_EPOCHS     = 5
BATCH_SIZE   = 32
ADAMW_LR     = 1e-4
ADAMW_WD     = 1e-4
N_GROUPS     = 9
COLLAPSE_THR = 0.01
N_DARK_MITIG = 200
DRO_ETA      = 0.1
DAGC_CLIP    = 1.0
ADV_LAMBDA   = 1.0
SMOTE_K      = 5
NC_NG_PAPER  = 203 / 2168

INTERVENTIONS = [
    'Baseline',
    'Real Oversample',
    'Oversampling + DAGC',
    'Group DRO',
    'Adversarial Debiasing',
    'SMOTE',
]
ARCH_ORDER = ['CLIP ViT-L/14', 'ViT-B/16', 'ResNet-50', 'DINOv2-Base']

# ── Dataset auto-discovery ────────────────────────────────────────────────────
_fitz_csv = None
for _root, _dirs, _files in os.walk('/kaggle/input'):
    for _f in _files:
        if _f.endswith('.csv') and 'fitzpatrick' in _f.lower():
            _fitz_csv = os.path.join(_root, _f); break
    if _fitz_csv: break

fitz_csv     = _fitz_csv or '/kaggle/input/fitzpatrick17k/fitzpatrick17k.csv'
fitz_img_dir = os.path.dirname(fitz_csv)
print(f"CSV: {fitz_csv}")

df = pd.read_csv(fitz_csv)
df = df[df['fitzpatrick_scale'] > 0]
image_files = {}
for _r, _, _fs in os.walk(fitz_img_dir):
    for _f in _fs:
        if _f.endswith(('.jpg','.png')):
            image_files[_f.rsplit('.',1)[0]] = os.path.join(_r, _f)
print(f"Images found: {len(image_files)}")

df['local_path'] = df['md5hash'].map(image_files)
df = df[df['local_path'].notna()].copy()
df['skin_group'] = df['fitzpatrick_scale'].apply(
    lambda x: 'light' if x<=2 else ('medium' if x<=4 else 'dark'))

np.random.seed(BASE_SEED)
# Use ALL available images — no cap. The paper used the full Fitzpatrick17k
# partitions; capping at 1,000 light-skin images leaves ~90 benign examples,
# which is not enough for the linear probe to learn the class (benign_acc → 0).
light_df  = df[df['skin_group']=='light'].copy()
medium_df = df[df['skin_group']=='medium'].copy()
dark_df   = df[df['skin_group']=='dark'].copy()
print(f"Splits: light={len(light_df)}, medium={len(medium_df)}, dark={len(dark_df)}")

le = LabelEncoder()
le.fit(list(light_df['three_partition_label']) +
       list(medium_df['three_partition_label']) +
       list(dark_df['three_partition_label']))
BENIGN_IDX = int(list(le.classes_).index('benign'))
MINORITY_G = 2 * 3 + BENIGN_IDX   # dark(=2) x 3 classes + benign_idx
print(f"Classes: {le.classes_}  |  BENIGN_IDX={BENIGN_IDX}  |  MINORITY_G={MINORITY_G}")

# ── Helpers ───────────────────────────────────────────────────────────────────
def wilson_ci(k, n, z=1.96):
    if n == 0: return 0.0, 0.0
    p = k/n; d = 1+z**2/n
    c = (p+z**2/(2*n))/d
    m = (z*np.sqrt(p*(1-p)/n+z**2/(4*n**2)))/d
    return max(0.0,c-m), min(1.0,c+m)

def eval_metrics(probs, preds, labels):
    mask = labels == BENIGN_IDX
    k = int((preds[mask]==labels[mask]).sum()) if mask.sum()>0 else 0
    n = int(mask.sum())
    acc = k/n if n>0 else 0.0
    lo, hi = wilson_ci(k, n)
    try:   auc = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except: auc = float('nan')
    return acc, lo, hi, auc

def load_imgs(dataframe):
    imgs, lbls = [], []
    for _, row in dataframe.iterrows():
        try:
            img = Image.open(row['local_path']).convert('RGB').resize((224,224))
            imgs.append(img)
            lbls.append(le.transform([row['three_partition_label']])[0])
        except: pass
    return imgs, np.array(lbls)

# ── Backbone loaders ──────────────────────────────────────────────────────────
def load_clip():
    m = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()
    p = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
    @torch.no_grad()
    def extract(imgs, bs=32):
        out = []
        for i in range(0, len(imgs), bs):
            inp = p(images=imgs[i:i+bs], return_tensors="pt", padding=True)
            inp = {k: v.to(device) for k, v in inp.items()}
            f = m.get_image_features(**inp)
            f = f / f.norm(dim=-1, keepdim=True)
            out.append(f.cpu().numpy())
        return np.vstack(out)
    return m, extract, 768

def load_vit_b16():
    m = ViTModel.from_pretrained("google/vit-base-patch16-224").to(device).eval()
    p = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
    @torch.no_grad()
    def extract(imgs, bs=32):
        out = []
        for i in range(0, len(imgs), bs):
            inp = p(images=imgs[i:i+bs], return_tensors="pt")
            inp = {k: v.to(device) for k, v in inp.items()}
            f = m(**inp).last_hidden_state[:, 0, :]
            f = f / f.norm(dim=-1, keepdim=True)
            out.append(f.cpu().numpy())
        return np.vstack(out)
    return m, extract, 768

def load_resnet50():
    base = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1).to(device).eval()
    feat_m = nn.Sequential(*list(base.children())[:-1])
    tfm = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    @torch.no_grad()
    def extract(imgs, bs=32):
        out = []
        for i in range(0, len(imgs), bs):
            batch = torch.stack([tfm(im) for im in imgs[i:i+bs]]).to(device)
            f = feat_m(batch).squeeze(-1).squeeze(-1)
            f = f / f.norm(dim=-1, keepdim=True)
            out.append(f.cpu().numpy())
        return np.vstack(out)
    return feat_m, extract, 2048

def load_dinov2():
    m = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14',
                       pretrained=True).to(device).eval()
    tfm = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    @torch.no_grad()
    def extract(imgs, bs=32):
        out = []
        for i in range(0, len(imgs), bs):
            batch = torch.stack([tfm(im) for im in imgs[i:i+bs]]).to(device)
            f = m(batch)
            f = f / f.norm(dim=-1, keepdim=True)
            out.append(f.cpu().numpy())
        return np.vstack(out)
    return m, extract, 768

ARCH_REGISTRY = [
    ('CLIP ViT-L/14', load_clip,    768),
    ('ViT-B/16',      load_vit_b16, 768),
    ('ResNet-50',     load_resnet50, 2048),
    ('DINOv2-Base',   load_dinov2,  768),
]

# ── Intervention heads ────────────────────────────────────────────────────────
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam): ctx.lam=lam; return x.clone()
    @staticmethod
    def backward(ctx, grad):  return -ctx.lam*grad, None

class AdvHead(nn.Module):
    def __init__(self, fd, nc=3, ns=3, lam=ADV_LAMBDA):
        super().__init__()
        self.clf  = nn.Linear(fd, nc)
        self.disc = nn.Sequential(nn.Linear(fd,64), nn.ReLU(), nn.Linear(64,ns))
        self.lam  = lam
    def forward(self, x):
        return self.clf(x), self.disc(GradReverse.apply(x, self.lam))

def apply_smote(X, y, seed):
    k = min(SMOTE_K, int((y==BENIGN_IDX).sum())-1)
    if k < 1: return X, y
    X_r, y_r = _SMOTE(k_neighbors=k, random_state=seed).fit_resample(X, y)
    norms = np.linalg.norm(X_r, axis=1, keepdims=True)
    return X_r / np.where(norms==0, 1, norms), y_r

# ── Trainers ──────────────────────────────────────────────────────────────────
def run_erm(fd, tr_f, tr_y, tr_s, te_f, te_y, seed,
            oversample=False, dagc=False,
            aug_dark_f=None, aug_dark_y=None):
    torch.manual_seed(seed); np.random.seed(seed)
    X, y, s = tr_f.copy(), tr_y.copy(), tr_s.copy()
    if oversample and aug_dark_f is not None and len(aug_dark_f) > 0:
        # Sample N_DARK_MITIG dark images on top of the light-only training set
        rng = np.random.RandomState(seed)
        idx = rng.choice(len(aug_dark_f), size=N_DARK_MITIG, replace=(len(aug_dark_f)<N_DARK_MITIG))
        aug_s = np.full(len(idx), 2, dtype=int)
        X = np.vstack([X, aug_dark_f[idx]])
        y = np.concatenate([y, aug_dark_y[idx]])
        s = np.concatenate([s, aug_s])
    head = nn.Linear(fd,3).to(device)
    nn.init.xavier_uniform_(head.weight); nn.init.zeros_(head.bias)
    opt  = optim.AdamW(head.parameters(), lr=ADAMW_LR, weight_decay=ADAMW_WD)
    crit = nn.CrossEntropyLoss()
    gen  = torch.Generator(); gen.manual_seed(seed)
    loader = DataLoader(TensorDataset(torch.tensor(X,dtype=torch.float32),
                                      torch.tensor(y,dtype=torch.long),
                                      torch.tensor(s,dtype=torch.long)),
                        batch_size=BATCH_SIZE, shuffle=True, generator=gen)
    for _ in range(N_EPOCHS):
        head.train()
        for xb,yb,sb in loader:
            xb,yb,sb = xb.to(device),yb.to(device),sb.to(device)
            opt.zero_grad(); loss=crit(head(xb),yb); loss.backward()
            if dagc: nn.utils.clip_grad_norm_(head.parameters(), DAGC_CLIP)
            opt.step()
    head.eval()
    with torch.no_grad():
        probs=torch.softmax(head(torch.tensor(te_f,dtype=torch.float32).to(device)),-1).cpu().numpy()
    return eval_metrics(probs, np.argmax(probs,1), te_y)

def run_smote(fd, tr_f, tr_y, te_f, te_y, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    Xs, ys = apply_smote(tr_f, tr_y, seed)
    head = nn.Linear(fd,3).to(device)
    nn.init.xavier_uniform_(head.weight); nn.init.zeros_(head.bias)
    opt  = optim.AdamW(head.parameters(), lr=ADAMW_LR, weight_decay=ADAMW_WD)
    crit = nn.CrossEntropyLoss()
    gen  = torch.Generator(); gen.manual_seed(seed)
    loader = DataLoader(TensorDataset(torch.tensor(Xs,dtype=torch.float32),
                                      torch.tensor(ys,dtype=torch.long)),
                        batch_size=BATCH_SIZE, shuffle=True, generator=gen)
    for _ in range(N_EPOCHS):
        head.train()
        for xb,yb in loader:
            xb,yb=xb.to(device),yb.to(device)
            opt.zero_grad(); crit(head(xb),yb).backward(); opt.step()
    head.eval()
    with torch.no_grad():
        probs=torch.softmax(head(torch.tensor(te_f,dtype=torch.float32).to(device)),-1).cpu().numpy()
    return eval_metrics(probs, np.argmax(probs,1), te_y)

def run_dro(fd, tr_f, tr_y, tr_g, te_f, te_y, seed, eta=DRO_ETA):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); np.random.seed(seed)
    head = nn.Linear(fd,3).to(device)
    nn.init.xavier_uniform_(head.weight); nn.init.zeros_(head.bias)
    opt  = optim.AdamW(head.parameters(), lr=ADAMW_LR, weight_decay=ADAMW_WD)
    crit = nn.CrossEntropyLoss(reduction='none')
    gen  = torch.Generator(); gen.manual_seed(seed)
    loader = DataLoader(TensorDataset(torch.tensor(tr_f,dtype=torch.float32),
                                      torch.tensor(tr_y,dtype=torch.long),
                                      torch.tensor(tr_g,dtype=torch.long)),
                        batch_size=BATCH_SIZE, shuffle=True, generator=gen)
    q = torch.ones(N_GROUPS, device=device)/N_GROUPS
    for _ in range(N_EPOCHS):
        head.train()
        for xb,yb,gb in loader:
            xb,yb,gb=xb.to(device),yb.to(device),gb.to(device)
            losses=crit(head(xb),yb)
            tl=torch.zeros(1,device=device,requires_grad=True).squeeze()
            bgl=torch.zeros(N_GROUPS,device=device)
            for g in range(N_GROUPS):
                mg=(gb==g)
                if mg.sum()>0:
                    gl=losses[mg].mean(); tl=tl+q[g].detach()*gl; bgl[g]=gl.detach()
            opt.zero_grad(); tl.backward(); opt.step()
            with torch.no_grad():
                act=bgl>0
                q[act]=q[act]*torch.exp(torch.tensor(eta,device=device)*bgl[act])
                q=q/q.sum()
    head.eval()
    with torch.no_grad():
        probs=torch.softmax(head(torch.tensor(te_f,dtype=torch.float32).to(device)),-1).cpu().numpy()
    collapse = bool(q[MINORITY_G].item()<COLLAPSE_THR)
    acc,lo,hi,auc = eval_metrics(probs, np.argmax(probs,1), te_y)
    return acc, lo, hi, auc, collapse, float(q[MINORITY_G].item())

def run_adv(fd, tr_f, tr_y, tr_s, te_f, te_y, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    head = AdvHead(fd).to(device)
    opt  = optim.AdamW(head.parameters(), lr=ADAMW_LR, weight_decay=ADAMW_WD)
    cc   = nn.CrossEntropyLoss(); dc = nn.CrossEntropyLoss()
    gen  = torch.Generator(); gen.manual_seed(seed)
    loader = DataLoader(TensorDataset(torch.tensor(tr_f,dtype=torch.float32),
                                      torch.tensor(tr_y,dtype=torch.long),
                                      torch.tensor(tr_s,dtype=torch.long)),
                        batch_size=BATCH_SIZE, shuffle=True, generator=gen)
    for _ in range(N_EPOCHS):
        head.train()
        for xb,yb,sb in loader:
            xb,yb,sb=xb.to(device),yb.to(device),sb.to(device)
            co,do=head(xb); opt.zero_grad(); (cc(co,yb)+dc(do,sb)).backward(); opt.step()
    head.eval()
    with torch.no_grad():
        co,_=head(torch.tensor(te_f,dtype=torch.float32).to(device))
        probs=torch.softmax(co,-1).cpu().numpy()
    return eval_metrics(probs, np.argmax(probs,1), te_y)

# ── Main loop ─────────────────────────────────────────────────────────────────
csv_path = '/kaggle/working/intervention_matrix_results.csv'
all_rows = []
RAND_AUC_BY_ARCH = {}

print("\n"+"="*72)
print("INTERVENTION MATRIX: 4 architectures x 6 interventions x 5 seeds")
print("="*72)

for arch_name, load_fn, feat_dim in ARCH_REGISTRY:
    print(f"\n{'─'*60}\nARCHITECTURE: {arch_name}  (feat_dim={feat_dim})\n{'─'*60}")

    # Load model + extract features
    _, extract_fn, _ = load_fn()
    light_imgs,  light_y  = load_imgs(light_df)
    medium_imgs, medium_y = load_imgs(medium_df)
    dark_imgs,   dark_y   = load_imgs(dark_df)
    light_feats  = extract_fn(light_imgs)
    medium_feats = extract_fn(medium_imgs)
    dark_feats   = extract_fn(dark_imgs)
    print(f"  Features: light={light_feats.shape}, medium={medium_feats.shape}, dark={dark_feats.shape}")

    # FIX 1: Training set = light-skin only (demographically-aware split)
    base_f   = light_feats.copy()
    base_y   = light_y.copy()
    skin_ids = np.zeros(len(light_feats), dtype=int)   # group 0 = light
    base_g   = (skin_ids * 3 + base_y).astype(int)

    # DRO needs 3 skin groups to make sense — light + medium only (no dark in train)
    dro_f    = np.vstack([light_feats, medium_feats])
    dro_y    = np.concatenate([light_y, medium_y])
    dro_s    = np.concatenate([np.zeros(len(light_feats), dtype=int),
                                np.ones(len(medium_feats), dtype=int)])
    dro_g    = (dro_s * 3 + dro_y).astype(int)

    # FIX 2: Test set = ALL dark-skin images (don't carve 200 out for training)
    test_f     = dark_feats
    test_y_arr = dark_y
    nc_ng = (test_y_arr == BENIGN_IDX).mean()
    print(f"  Test: n={len(test_f)}, nc/Ng={nc_ng:.4f} (paper: {NC_NG_PAPER:.4f})")

    # FIX 3: Random-split AUC — same AdamW linear probe (not LogisticRegression)
    # Pool light + dark, do a 75/25 stratified split, train the same 5-epoch probe
    pool_f  = np.vstack([light_feats, dark_feats])
    pool_y  = np.concatenate([light_y, dark_y])
    pool_s  = np.concatenate([np.zeros(len(light_feats), dtype=int),
                               np.full(len(dark_feats), 2, dtype=int)])
    tri_r, tei_r = train_test_split(np.arange(len(pool_y)), test_size=0.25,
                                    stratify=pool_y, random_state=BASE_SEED)
    # Use seed=BASE_SEED; we only need one estimate of the random-split ceiling
    _rand_head = nn.Linear(feat_dim, 3).to(device)
    nn.init.xavier_uniform_(_rand_head.weight); nn.init.zeros_(_rand_head.bias)
    _rand_opt  = optim.AdamW(_rand_head.parameters(), lr=ADAMW_LR, weight_decay=ADAMW_WD)
    _rand_crit = nn.CrossEntropyLoss()
    _rand_gen  = torch.Generator(); _rand_gen.manual_seed(BASE_SEED)
    _rand_loader = DataLoader(
        TensorDataset(torch.tensor(pool_f[tri_r], dtype=torch.float32),
                      torch.tensor(pool_y[tri_r], dtype=torch.long)),
        batch_size=BATCH_SIZE, shuffle=True, generator=_rand_gen)
    for _ in range(N_EPOCHS):
        _rand_head.train()
        for xb, yb in _rand_loader:
            xb, yb = xb.to(device), yb.to(device)
            _rand_opt.zero_grad(); _rand_crit(_rand_head(xb), yb).backward(); _rand_opt.step()
    _rand_head.eval()
    with torch.no_grad():
        _rp = torch.softmax(_rand_head(torch.tensor(pool_f[tei_r], dtype=torch.float32).to(device)), -1).cpu().numpy()
    rand_auc = roc_auc_score(pool_y[tei_r], _rp, multi_class='ovr', average='macro')
    del _rand_head, _rand_opt, _rand_loader
    RAND_AUC_BY_ARCH[arch_name] = rand_auc
    print(f"  Random-split AUC (AdamW probe): {rand_auc:.4f}")

    # Run all 6 interventions x 5 seeds
    for interv in INTERVENTIONS:
        print(f"\n  [{arch_name}] {interv}")
        for seed in SEEDS:
            print(f"    seed={seed}...", end=' ', flush=True)
            collapse=False; min_wt=float('nan')

            if interv == 'Baseline':
                acc,lo,hi,auc = run_erm(feat_dim,base_f,base_y,skin_ids,test_f,test_y_arr,seed)
            elif interv == 'Real Oversample':
                acc,lo,hi,auc = run_erm(feat_dim,base_f,base_y,skin_ids,test_f,test_y_arr,seed,oversample=True,aug_dark_f=dark_feats,aug_dark_y=dark_y)
            elif interv == 'Oversampling + DAGC':
                acc,lo,hi,auc = run_erm(feat_dim,base_f,base_y,skin_ids,test_f,test_y_arr,seed,oversample=True,dagc=True,aug_dark_f=dark_feats,aug_dark_y=dark_y)
            elif interv == 'Group DRO':
                acc,lo,hi,auc,collapse,min_wt = run_dro(feat_dim,dro_f,dro_y,dro_g,test_f,test_y_arr,seed)
            elif interv == 'Adversarial Debiasing':
                acc,lo,hi,auc = run_adv(feat_dim,base_f,base_y,skin_ids,test_f,test_y_arr,seed)
            elif interv == 'SMOTE':
                acc,lo,hi,auc = run_smote(feat_dim,base_f,base_y,test_f,test_y_arr,seed)

            sgg = rand_auc - auc
            row = dict(arch=arch_name, intervention=interv, seed=seed,
                       benign_acc=float(acc), benign_ci_lo=float(lo), benign_ci_hi=float(hi),
                       demo_auc=float(auc), sgg=float(sgg),
                       weight_collapse=bool(collapse), final_min_wt=float(min_wt))
            all_rows.append(row)
            print(f"benign={acc:.3f}  auc={auc:.4f}  sgg={sgg:+.4f}  collapse={collapse}")

        pd.DataFrame(all_rows).to_csv(csv_path, index=False)
        print(f"  → CSV saved ({len(all_rows)} rows)")

    del light_feats, medium_feats, dark_feats
    torch.cuda.empty_cache()

print("\n✓ All runs complete.")

# ── Figure ────────────────────────────────────────────────────────────────────
df_res = pd.read_csv(csv_path)
agg = df_res.groupby(['arch','intervention']).agg(
    benign_mean=('benign_acc','mean'), benign_std=('benign_acc','std'),
    demo_auc_mean=('demo_auc','mean')).reset_index()

agg['arch']         = pd.Categorical(agg['arch'],         categories=ARCH_ORDER,    ordered=True)
agg['intervention'] = pd.Categorical(agg['intervention'], categories=INTERVENTIONS, ordered=True)
agg = agg.sort_values(['arch','intervention'])
benign_mat = agg.pivot(index='intervention', columns='arch', values='benign_mean').loc[INTERVENTIONS, ARCH_ORDER]
auc_mat    = agg.pivot(index='intervention', columns='arch', values='demo_auc_mean').loc[INTERVENTIONS, ARCH_ORDER]

fig, axes = plt.subplots(1, 2, figsize=(16,6))
fig.suptitle('Intervention Matrix — All 4 Architectures (mean, 5 seeds, frozen linear probe)\n'
             'Fitzpatrick17k, dark-skin test set (FST V-VI)',
             fontsize=13, fontweight='bold')
for ax_i, (mat, title, fmt, cmap) in enumerate([
    (benign_mat*100, 'Dark-Skin Benign Accuracy (%)', '{:.1f}%', 'RdYlGn'),
    (auc_mat,        'Demographic AUC',               '{:.3f}',  'Blues'),
]):
    ax = axes[ax_i]
    im = ax.imshow(mat.values, cmap=cmap, aspect='auto',
                   vmin=mat.values.min(), vmax=mat.values.max())
    plt.colorbar(im, ax=ax, shrink=0.85)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xticks(range(len(ARCH_ORDER)));   ax.set_xticklabels(ARCH_ORDER, rotation=20, ha='right', fontsize=9)
    ax.set_yticks(range(len(INTERVENTIONS))); ax.set_yticklabels(INTERVENTIONS, fontsize=9)
    for r in range(len(INTERVENTIONS)):
        for c in range(len(ARCH_ORDER)):
            v = mat.values[r,c]
            txt = fmt.format(v*(100 if '%' in fmt else 1))
            color = 'red' if (INTERVENTIONS[r]=='Group DRO' and ax_i==0) else 'black'
            ax.text(c, r, txt, ha='center', va='center', fontsize=8, fontweight='bold', color=color)
plt.tight_layout()
fig_path = '/kaggle/working/nb_intervention_matrix_figure.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"Figure saved: {fig_path}")

# ── Summary + LaTeX ───────────────────────────────────────────────────────────
print("\n=== SUMMARY TABLE (mean +/- SD, 5 seeds) ===")
print(f"{'Arch':>18} {'Intervention':>24} {'Benign Acc':>14} {'Demo AUC':>12} {'SGG':>10} {'Collapse':>10}")
print("-"*95)
for arch in ARCH_ORDER:
    for interv in INTERVENTIONS:
        sub = df_res[(df_res['arch']==arch)&(df_res['intervention']==interv)]
        ba=sub['benign_acc'].mean(); bas=sub['benign_acc'].std()
        au=sub['demo_auc'].mean();   aus=sub['demo_auc'].std()
        sg=sub['sgg'].mean()
        co=sub['weight_collapse'].mean() if interv=='Group DRO' else float('nan')
        co_str=f"{co:.0%} ({int(co*5)}/5)" if not np.isnan(co) else "—"
        print(f"{arch:>18} {interv:>24} {ba:.3f}+/-{bas:.3f}  {au:.4f}+/-{aus:.4f}  {sg:+.4f}  {co_str:>10}")
    print()

print("\n=== LaTeX SUPPLEMENTARY TABLE ROWS ===")
print("\\hline")
for arch in ARCH_ORDER:
    for i_i, interv in enumerate(INTERVENTIONS):
        sub = df_res[(df_res['arch']==arch)&(df_res['intervention']==interv)]
        ba=sub['benign_acc'].mean(); bas=sub['benign_acc'].std()
        au=sub['demo_auc'].mean();   aus=sub['demo_auc'].std()
        sg=sub['sgg'].mean();        sgs=sub['sgg'].std()
        co=sub['weight_collapse'].mean() if interv=='Group DRO' else None
        co_str=f"{int(co*5)}/5" if co is not None else "—"
        arch_str = arch if i_i==0 else ""
        print(f"{arch_str} & {interv} & {ba:.3f} $\\pm$ {bas:.3f} & "
              f"{au:.4f} $\\pm$ {aus:.4f} & {sg:+.4f} $\\pm$ {sgs:.4f} & {co_str} \\\\")
    print("\\hline")

print("\n=== INTERPRETATION FLAGS ===")
dro = df_res[df_res['intervention']=='Group DRO']
print(f"DRO benign_acc < 0.01: {(dro['benign_acc']<0.01).mean():.0%} ({int((dro['benign_acc']<0.01).sum())}/{len(dro)} runs)")
smote = df_res[df_res['intervention']=='SMOTE']
print(f"SMOTE mean benign_acc by arch:")
for arch in ARCH_ORDER:
    s=smote[smote['arch']==arch]
    print(f"  {arch:>18}: {s['benign_acc'].mean():.3f} +/- {s['benign_acc'].std():.3f}")

print("\n✓ Complete.")
print("Files: intervention_matrix_results.csv, nb_intervention_matrix_figure.png")
print("Upload figure + paste ALL output (LaTeX rows) to Claude.")
